In [15]:
# importing packages
import pyomo.environ as pyo
import numpy as np
from matplotlib import pyplot as plt
import math
import scipy.stats as stats
from scipy.optimize import fsolve, bisect, minimize
import pandas as pd
# import shedlovsky_conductivity_pyomo as eq
import three_salt_conductivity as eq

# Three-Salt System Conductivity Predictions

## Load Data

In [5]:
data = pd.read_csv("./Data/Three_salts_cation_ratio.csv")
NaCl_conc_raw = data["NaCl_conc"]
CaCl2_conc_raw = data["CaCl2_conc"]
LaCl3_conc_raw = data["LaCl3_conc"]

print(LaCl3_conc_raw)

# number of data points
ndata = len(NaCl_conc_raw)

0     2
1     4
2     6
3     8
4    10
5    12
6    14
7    16
8    18
9    20
Name: LaCl3_conc, dtype: int64


## Determining the best cation ratio

### MSA Transport Model Constants in SI Unit

In [14]:
# temperature
temp = 298.15

# diffusion coefficients at infinite dilution
Diff_Na = 1.334 * 10**(-9)
Diff_Cl = 2.032 * 10**(-9)
Diff_Ca = 0.792 * 10**(-9)
Diff_La = 0.619 * 10**(-9)

# valencies
z_Na = 1
z_Cl = -1
z_Ca = 2
z_La = 3

# measuring voltage and probe distance
voltage = 25 * 10**(-3)
distance = 9 * 10**(-6)

# hard sphere diameters
sigma_Cl = 181 * 10**(-12)
sigma_Ca = 100 * 10**(-12)
sigma_La = 103.2 * 10**(-12)
sigma_Na = 102 * 10**(-12)

# solvent viscosity
eta_msa = 0.89 * 10**(-3)

# electric field
E_field = voltage/distance

# relative permittivity
epsilon = 78.4

In [7]:
diffusion_coeff = [Diff_La, Diff_Na, Diff_Cl] # diffusion coefficients of La, Ca, Na, and Cl, respectively
valency = [z_La, z_Na, z_Cl] # valencies of La, Ca, Na, and Cl, respectively
diameter = [sigma_La, sigma_Na, sigma_Cl] # effective ionic diameters of La, Ca, Na, and Cl, respectively

### Variant Shedlovsky model

In [8]:
# solvent viscosity in poise
eta = 8.903*10**(-3)

# limiting equivalent conductivity of salts in cm^2.Siemen.equiv^(-1)
lambda_0_NaCl = 126.45 
lambda_0_CaCl2 = 135.85
lambda_0_LaCl3 = 145.9

# limiting equivalent conductivity of ions in cm^2.Siemen.equiv^(-1)
lambda_0_Na = 50.10 
lambda_0_Ca = 59.5
lambda_0_La = 69.7
lambda_0_Cl = 76.35

# distance from an ion within which no other ion can penetrate in cm
a_NaCl = 4*10**(-8)
a_CaCl2 = 4.31*10**(-8)
a_LaCl3 = 4.9*10**(-8)
a = (a_NaCl + a_LaCl3)/2
# a = (a_NaCl + a_CaCl2 + a_LaCl3)/3

# salt concentration in M
conc_NaCl_data = [NaCl_conc_raw[i]*10**(-3) for i in range(ndata)]
conc_CaCl2_data = [CaCl2_conc_raw[i]*10**(-3) for i in range(ndata)]
conc_LaCl3_data = [LaCl3_conc_raw[i]*10**(-3) for i in range(ndata)]

# conc_NaCl = {i: NaCl_conc[i]*10**(-3) for i in range(ndata)}
# conc_CaCl2 = {i: CaCl2_conc[i]*10**(-3) for i in range(ndata)}
# conc_LaCl3 = {i: LaCl3_conc[i]*10**(-3) for i in range(ndata)}

In [18]:
data4 = pd.read_csv("./Data/NaCl_LaCl3_rearranged.csv")
NaCl_conc22 = data4["NaCl conc2"]
LaCl3_conc22 = data4["LaCl3 conc2"]
cond_exp32 = data4["Cal bulk cond2"]
print(cond_exp32)

conc_NaCl_shed = [3*conc_NaCl_data[i] for i in range(ndata)]
conc_LaCl3_shed = [1*conc_LaCl3_data[i] for i in range(ndata)]

conc_NaCl = [3*NaCl_conc_raw[i] for i in range(ndata)]
conc_LaCl3 = [1*LaCl3_conc_raw[i] for i in range(ndata)]

variant_shed_cond_NaCl = eq.variant_shedlovsky(conc_NaCl_shed, temp, epsilon, eta, lambda_0_NaCl, a, z_Na, z_Cl, 
                             lambda_0_Na, lambda_0_Cl)
variant_shed_cond_LaCl3 = eq.variant_shedlovsky(conc_LaCl3_shed, temp, epsilon, eta, lambda_0_LaCl3, a, z_La, z_Cl, 
                             lambda_0_La, lambda_0_Cl)

variant_shed_cond = [variant_shed_cond_NaCl[i] + variant_shed_cond_LaCl3[i] for i in range(ndata)]
msa_cond = eq.msa_transport(valency, diameter, diffusion_coeff, temp, voltage, distance, eta_msa, epsilon, conc_LaCl3, conc_NaCl)
sse = sum((msa_cond[i] - variant_shed_cond[i])**2 for i in range(ndata))
print(variant_shed_cond)
print(msa_cond)
print(sse)

0     1467.883721
1     2752.339767
2     4000.965581
3     5209.484651
4     6390.033023
5     7584.451163
6     8697.731163
7     9872.038140
8    10984.624650
9    12124.488370
Name: Cal bulk cond2, dtype: float64
Relaxation correction: [-3.84021160e-11 -6.94462386e-12 -2.26733699e-11]
Hydrodynamic correction: [-0.47872796 -0.22184151 -0.14000104]
omega_bar: 293668709070.8091
Alpha: [0.0, 242876329097.13528, 388092787538.26306]
Mew: [0.5        0.16666667 0.33333333]
Transport number: [0.25596141 0.18387319 0.5601654 ]
Omega: [1.50335711e+11 3.23986816e+11 4.93509152e+11]
ksi: [[ 1.69629658 -0.98749428 -0.23296339]
 [ 0.78711213  1.6842391  -1.40795025]
 [ 0.51673602  0.63912188  1.0534202 ]]
[1483.642091052549, 2864.234360770388, 4187.946948739507, 5470.196917580431, 6719.481069605823, 7941.301552570567, 9139.551037965753, 10317.145022129693, 11476.356410740711, 12619.010832978776]
[1495.3193815131024, 2884.0660648526355, 4207.804330221462, 5481.1270717425305, 6712.364194540472, 79

In [19]:
def objective_function(x):
    conc_NaCl_shed = [x[0]*conc_NaCl_data[i] for i in range(ndata)]
    conc_LaCl3_shed = [x[1]*conc_LaCl3_data[i] for i in range(ndata)]
    conc_NaCl = [x[0]*NaCl_conc_raw[i] for i in range(ndata)]
    conc_LaCl3 = [x[1]*LaCl3_conc_raw[i] for i in range(ndata)]
    
    variant_shed_cond_NaCl = eq.variant_shedlovsky(conc_NaCl_shed, temp, epsilon, eta, lambda_0_NaCl, a, z_Na, z_Cl, 
                                 lambda_0_Na, lambda_0_Cl)
    variant_shed_cond_LaCl3 = eq.variant_shedlovsky(conc_LaCl3_shed, temp, epsilon, eta, lambda_0_LaCl3, a, z_La, z_Cl, 
                                 lambda_0_La, lambda_0_Cl)
    
    variant_shed_cond = [variant_shed_cond_NaCl[i] + variant_shed_cond_LaCl3[i] for i in range(ndata)]
    msa_cond = eq.msa_transport(valency, diameter, diffusion_coeff, temp, voltage, distance, eta_msa, epsilon, conc_LaCl3, conc_NaCl)
    
    # return sum((msa_cond[i] - cond_exp32[i])**2 for i in range(ndata))
    return sum((msa_cond[i] - variant_shed_cond[i])**2 for i in range(ndata))

In [22]:
cation_ratio = [1, 1]

result = minimize(objective_function, cation_ratio, bounds = [(1, 5), (1, 1)])

print(result.x)
print(result.fun)

Relaxation correction: [-2.95605070e-11 -4.68449807e-12 -2.33415048e-11]
Hydrodynamic correction: [-0.42954208 -0.19913596 -0.12604011]
omega_bar: 260788916222.98755
Alpha: [0.0, 280185123994.9613, 378474712351.47736]
Mew: [0.64285714 0.07142857 0.28571429]
Transport number: [0.37058471 0.08873811 0.54067718]
Omega: [1.50335711e+11 3.23986816e+11 4.93509152e+11]
ksi: [[ 1.50778913 -0.58944861 -0.27849707]
 [ 0.69964128  2.68345218 -1.89174574]
 [ 0.45931175  0.65539633  1.09955484]]
Relaxation correction: [-2.95605070e-11 -4.68449808e-12 -2.33415047e-11]
Hydrodynamic correction: [-0.42954208 -0.19913596 -0.12604011]
omega_bar: 260788916434.3576
Alpha: [0.0, 280185123714.45715, 378474712443.7822]
Mew: [0.64285714 0.07142857 0.28571429]
Transport number: [0.37058471 0.08873811 0.54067718]
Omega: [1.50335711e+11 3.23986816e+11 4.93509152e+11]
ksi: [[ 1.50778913 -0.58944861 -0.27849707]
 [ 0.69964128  2.68345216 -1.89174573]
 [ 0.45931175  0.65539633  1.09955484]]
Relaxation correction: [-

In [12]:
m = pyo.ConcreteModel()

# define the data index set
m.I = pyo.RangeSet(0, ndata - 1)

# define input and output variables
m.Na_ratio = pyo.Var(within=pyo.NonNegativeReals, initialize=1)
m.Ca_ratio = pyo.Var(within=pyo.NonNegativeReals, initialize=1)
m.La_ratio = pyo.Var(within=pyo.NonNegativeReals, initialize=1)

m.conc_NaCl = pyo.Param(m.I, initialize=conc_NaCl)
m.conc_CaCl2 = pyo.Param(m.I, initialize=conc_CaCl2)
m.conc_LaCl3 = pyo.Param(m.I, initialize=conc_LaCl3)

m.variant_shed_cond = pyo.Var(m.I, within=pyo.NonNegativeReals)
m.msa_cond = pyo.Var(m.I, within=pyo.NonNegativeReals)

# variant Shedlovsky conductivity predictions
def variant_shedlovsky_salt_1(m, i):
    return eq.variant_shedlovsky(m.Na_ratio*m.conc_NaCl[i], temp, epsilon, eta, lambda_0_NaCl, a, z_Na, z_Cl, 
                                 lambda_0_Na, lambda_0_Cl)

# variant Shedlovsky conductivity predictions
def variant_shedlovsky_salt_2(m, i):
    return eq.variant_shedlovsky(m.Ca_ratio*m.conc_CaCl2[i], temp, epsilon, eta, lambda_0_CaCl2, a, z_Ca, z_Cl, 
                                 lambda_0_Ca, lambda_0_Cl)

# variant Shedlovsky conductivity predictions
def variant_shedlovsky_salt_3(m, i):
    return eq.variant_shedlovsky(m.La_ratio*m.conc_LaCl3[i], temp, epsilon, eta, lambda_0_LaCl3, a, z_La, z_Cl, 
                                 lambda_0_La, lambda_0_Cl)

m.salt_1_conductivity = pyo.Expression(m.I, rule=variant_shedlovsky_salt_1)
m.salt_2_conductivity = pyo.Expression(m.I, rule=variant_shedlovsky_salt_2)
m.salt_3_conductivity = pyo.Expression(m.I, rule=variant_shedlovsky_salt_3)

@m.Constraint(m.I)
def bulk_conductivity_shedlovsky(m, i):
    return m.variant_shed_cond[i] == m.salt_1_conductivity[i] + m.salt_2_conductivity[i] + m.salt_3_conductivity[i]

def msa_conductivity_expr(m, i):
    return eq.msa_transport(valency, diameter, diffusion_coeff, temp, voltage, distance, eta, epsilon,
                            m.La_ratio * m.conc_LaCl3[i],
                            m.Ca_ratio * m.conc_CaCl2[i],
                            m.Na_ratio * m.conc_NaCl[i])

m.msa_calc = pyo.Expression(m.I, rule=msa_conductivity_expr)

@m.Constraint(m.I)
def bulk_conductivity_msa(m, i):
    return m.msa_cond[i] == m.msa_calc[i]


# @m.Constraint(m.I)
# def bulk_conductivity_msa(m, i):
#     return m.msa_cond[i] == eq.msa_transport(valency, diameter, diffusion_coeff, temp, voltage, distance, eta, epsilon, 
#                                           pyo.value(m.La_ratio)*m.conc_LaCl3[i], pyo.value(m.Ca_ratio)*m.conc_CaCl2[i], 
#                                           pyo.value(m.Na_ratio)*m.conc_NaCl[i])

m.objective = pyo.Objective(expr=sum((m.variant_shed_cond[i] - m.msa_cond[i])**2 for i in m.I), sense=pyo.minimize)

solver = pyo.SolverFactory('ipopt')
solver.solve(m, tee=True)

print(pyo.value(m.Na_ratio))
print(pyo.value(m.Ca_ratio))
print(pyo.value(m.La_ratio))

NameError: name 'conc_CaCl2' is not defined